In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.1 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on HEiDi

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit one Co₂SiO₄ crystal measured on HEiDi at MLZ at 300 K, 55 K and 2.5 K. Each temperature has
its own structure and its own reflections, and the three are fitted independently: each fit sees
only its own data and gives its own goodness of fit and uncertainties. At 2.5 K the cobalt
moments order, and the magnetic structure is fitted too.

The data are those of Sazonov et al., Acta Cryst. B 64 (2008) 661-668. The 2.5 K file has no
usable uncertainties, so they are estimated from the 55 K data of the same crystal (see the
case's provenance); the fitted 2.5 K uncertainties depend on that estimate.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

The independent fitting mode fits each experiment on its own, against the structure it links.

In [3]:
project = edi.Project(name='cosio_heidi')
project.analysis.fitting_mode = 'independent'

## Define the structures

Load the three structures from EasyDiffraction's collection, one per temperature. Each starts
from the published positions with a small displacement tensor on every site. The 2.5 K structure
also carries the magnetic space group Pnma.1 and the moment directions it allows, Co1 in any
direction and Co2 along b, with every moment at zero. The files mark what the fit frees; the fit
below frees it in stages, so start with all of it fixed.

In [4]:
path = edi.download_data('sc-neut-cwl_cosio-heidi-cosio_300k', destination='data')
project.structures.add_from_cif_path(path)
path = edi.download_data('sc-neut-cwl_cosio-heidi-cosio_55k', destination='data')
project.structures.add_from_cif_path(path)
path = edi.download_data('sc-neut-cwl_cosio-heidi-cosio_2k5', destination='data')
project.structures.add_from_cif_path(path)
for parameter in project.free_parameters:
    parameter.free = False

Data 'sc-neut-cwl_cosio-heidi-cosio_300k' is at data/sc-neut-cwl_cosio-heidi-cosio_300k.edi
Data 'sc-neut-cwl_cosio-heidi-cosio_55k' is at data/sc-neut-cwl_cosio-heidi-cosio_55k.edi
Data 'sc-neut-cwl_cosio-heidi-cosio_2k5' is at data/sc-neut-cwl_cosio-heidi-cosio_2k5.edi


In [5]:
project.display.structure('cosio_2k5')

View(html=<298,081 characters>, height=660)

## Define the experiments

Load the reflections of each temperature and link each dataset to its structure. HEiDi's
wavelength is 0.552 Å; the extinction is FullProf's anisotropic Larson model, with its six
coefficients starting at zero.

In [6]:
data_paths = {
    '300k': edi.download_data('sc-neut-cwl_cosio-heidi-heidi_300k-data', destination='data'),
    '55k': edi.download_data('sc-neut-cwl_cosio-heidi-heidi_55k-data', destination='data'),
    '2k5': edi.download_data('sc-neut-cwl_cosio-heidi-heidi_2k5-data', destination='data'),
}
for temperature, path in data_paths.items():
    project.experiments.add_from_data_path(
        name=f'heidi_{temperature}',
        data_path=path,
        sample_form='single crystal',
        beam_mode='constant wavelength',
        radiation_probe='neutron',
        scattering_type='bragg',
    )
    experiment = project.experiments[f'heidi_{temperature}']
    experiment.instrument.setup_wavelength = 0.552
    experiment.extinction.type = 'larson-anisotropic'
    experiment.linked_structures.create(structure_id=f'cosio_{temperature}', scale=10)

Data 'sc-neut-cwl_cosio-heidi-heidi_300k-data' is at data/sc-neut-cwl_cosio-heidi-heidi_300k.hkl
Data 'sc-neut-cwl_cosio-heidi-heidi_55k-data' is at data/sc-neut-cwl_cosio-heidi-heidi_55k.hkl
Data 'sc-neut-cwl_cosio-heidi-heidi_2k5-data' is at data/sc-neut-cwl_cosio-heidi-heidi_2k5.hkl


Names for the three structures and experiments.

In [7]:
structure_300k = project.structures['cosio_300k']
structure_55k = project.structures['cosio_55k']
structure_2k5 = project.structures['cosio_2k5']
experiment_300k = project.experiments['heidi_300k']
experiment_55k = project.experiments['heidi_55k']
experiment_2k5 = project.experiments['heidi_2k5']

## Perform the analysis

An experiment with nothing free is left out of a fit, so the first stages fit 300 K alone. Each
stage frees one new group: the structure's coordinates and displacement components are the ones
its site symmetry leaves free.

The 300 K reflections against the starting model:

In [8]:
project.display.pattern('heidi_300k')

View(html=<254,428 characters>, height=480)

### 300 K: scale

The scale brings the calculated intensities to the measured ones.

In [9]:
experiment_300k.linked_structures['cosio_300k'].scale.free = True

In [10]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,heidi_300k,linked_structure,cosio_300k,scale,10,0,-inf,inf,


In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        33.37        14.23
      1      0.012      15.51         3.07   78.4% ↓
  done · converged · 1 iters · reduced χ² 3.07 · wR(F²) 15.51 % · 0.02 s


View(html=<88,448 characters>, height=376)

In [12]:
project.display.pattern('heidi_300k')

View(html=<254,956 characters>, height=480)

### 300 K: coordinates

In [13]:
for site in structure_300k.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 12 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        15.51         3.09
      1      0.028      15.34         3.03   2.2% ↓
  done · converged · 1 iters · reduced χ² 3.03 · wR(F²) 15.34 % · 0.03 s


View(html=<103,051 characters>, height=376)

In [15]:
project.display.pattern('heidi_300k')

View(html=<255,417 characters>, height=480)

### 300 K: displacements

In [16]:
for tensor in structure_300k.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 40 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        15.34         3.08
      1      0.052       7.67         0.77   75.0% ↓
  done · converged · 1 iters · reduced χ² 0.77 · wR(F²) 7.67 % · 0.06 s


View(html=<139,590 characters>, height=376)

In [18]:
project.display.pattern('heidi_300k')

View(html=<255,433 characters>, height=480)

### 300 K: extinction

Larson's anisotropic extinction, its six coefficients starting at zero.

In [19]:
experiment_300k.extinction.coefficient_11.free = True
experiment_300k.extinction.coefficient_22.free = True
experiment_300k.extinction.coefficient_33.free = True
experiment_300k.extinction.coefficient_12.free = True
experiment_300k.extinction.coefficient_13.free = True
experiment_300k.extinction.coefficient_23.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         7.67         0.77
      1      0.045       6.15         0.50   35.6% ↓
  done · converged · 1 iters · reduced χ² 0.50 · wR(F²) 6.15 % · 0.05 s


View(html=<147,415 characters>, height=376)

In [21]:
project.display.pattern('heidi_300k')

View(html=<255,559 characters>, height=480)

### 55 K: scale

The 300 K parameters stay free, so the 300 K experiment is fitted again from where it ended.

In [22]:
experiment_55k.linked_structures['cosio_55k'].scale.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.028       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        36.92        69.94
      1      0.012       8.90         4.06   94.2% ↓
  done · converged · 2 iters · reduced χ² 2.21 · wR(F²) 8.38 % · 0.05 s


View(html=<139,623 characters>, height=376)

In [24]:
project.display.pattern('heidi_55k')

View(html=<237,245 characters>, height=480)

### 55 K: coordinates

In [25]:
for site in structure_55k.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.028       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 12 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         8.90         4.09
      1      0.018       8.78         3.98   2.7% ↓
  done · converged · 2 iters · reduced χ² 2.17 · wR(F²) 8.28 % · 0.06 s


View(html=<153,464 characters>, height=376)

In [27]:
project.display.pattern('heidi_55k')

View(html=<237,334 characters>, height=480)

### 55 K: extinction

At 55 K the displacements are small. Freed before the extinction, they would take its place and
Co1's tensor would stop being positive definite, so the extinction comes before them here.

In [28]:
experiment_55k.extinction.coefficient_11.free = True
experiment_55k.extinction.coefficient_22.free = True
experiment_55k.extinction.coefficient_33.free = True
experiment_55k.extinction.coefficient_12.free = True
experiment_55k.extinction.coefficient_13.free = True
experiment_55k.extinction.coefficient_23.free = True

In [29]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.044       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 18 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         8.78         4.00
      1      0.018       6.58         2.25   43.9% ↓
      2      0.022       6.58         2.25   
  done · converged · 3 iters · reduced χ² 1.33 · wR(F²) 6.49 % · 0.08 s


View(html=<159,314 characters>, height=376)

In [30]:
project.display.pattern('heidi_55k')

View(html=<237,347 characters>, height=480)

### 55 K: displacements

In [31]:
for tensor in structure_55k.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [32]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.047       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.58         2.29
      1      0.040       3.79         0.76   66.7% ↓
  done · converged · 2 iters · reduced χ² 0.62 · wR(F²) 4.41 % · 0.11 s


View(html=<197,666 characters>, height=376)

In [33]:
project.display.pattern('heidi_55k')

View(html=<237,448 characters>, height=480)

### 2.5 K: scale

The nuclear structure first, with every moment still at zero, in the order 55 K needed.

In [34]:
experiment_2k5.linked_structures['cosio_2k5'].scale.free = True

In [35]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.045       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         3.79         0.76
      1      0.022       3.79         0.76   
edi fit: project cosio_heidi, 1424 pts fitted / 1424 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        39.10        63.48
      1      0.012      13.12         7.15   88.7% ↓
  done · converged · 3 iters · reduced χ² 2.72 · wR(F²) 8.82 % · 0.10 s


View(html=<190,333 characters>, height=376)

In [36]:
project.display.pattern('heidi_2k5')

View(html=<233,593 characters>, height=480)

### 2.5 K: coordinates

In [37]:
for site in structure_2k5.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [38]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.048       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         3.79         0.76
      1      0.028       3.79         0.76   
edi fit: project cosio_heidi, 1424 pts fitted / 1424 loaded, 12 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        13.12         7.21
      1      0.019      13.01         7.08   1.7% ↓
  done · converged · 3 iters · reduced χ² 2.69 · wR(F²) 8.75 % · 0.12 s


View(html=<204,465 characters>, height=376)

In [39]:
project.display.pattern('heidi_2k5')

View(html=<233,599 characters>, height=480)

### 2.5 K: extinction

In [40]:
experiment_2k5.extinction.coefficient_11.free = True
experiment_2k5.extinction.coefficient_22.free = True
experiment_2k5.extinction.coefficient_33.free = True
experiment_2k5.extinction.coefficient_12.free = True
experiment_2k5.extinction.coefficient_13.free = True
experiment_2k5.extinction.coefficient_23.free = True

In [41]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.050       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         3.79         0.76
      1      0.032       3.79         0.76   
edi fit: project cosio_heidi, 1424 pts fitted / 1424 loaded, 18 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        13.01         7.11
      1      0.021      11.88         5.93   16.6% ↓
      2      0.025      11.88         5.93   
  done · converged · 4 iters · reduced χ² 2.32 · wR(F²) 8.12 % · 0.13 s


View(html=<210,026 characters>, height=376)

In [42]:
project.display.pattern('heidi_2k5')

View(html=<233,646 characters>, height=480)

### 2.5 K: displacements

In [43]:
for tensor in structure_2k5.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [44]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.039       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         3.79         0.76
      1      0.032       3.79         0.76   
edi fit: project cosio_heidi, 1424 pts fitted / 1424 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        11.88         6.05
      1      0.041      10.51         4.74   21.6% ↓
  done · converged · 3 iters · reduced χ² 1.92 · wR(F²) 7.36 % · 0.14 s


View(html=<248,448 characters>, height=376)

Some reflections stay far from their measured intensity: (1 0 0), (0 0 1) and (1 1 0) among
them. Pnma forbids them for nuclear scattering, so with the moments at zero their calculated
intensity is zero; at 2.5 K they are magnetic.

In [45]:
project.display.pattern('heidi_2k5')

View(html=<233,824 characters>, height=480)

### 2.5 K: the magnetic moments

Free the moment components the magnetic space group allows. They start at zero, where the
magnetic intensity has no slope, so the fit starts them at 1 μB and says so.

In [46]:
structure_2k5.atom_site_moment['Co1'].crystalaxis_x.free = True
structure_2k5.atom_site_moment['Co1'].crystalaxis_y.free = True
structure_2k5.atom_site_moment['Co1'].crystalaxis_z.free = True
structure_2k5.atom_site_moment['Co2'].crystalaxis_y.free = True

In [47]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_heidi, 1624 pts fitted / 1624 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.15         0.50
      1      0.055       6.15         0.50   
edi fit: project cosio_heidi, 1465 pts fitted / 1465 loaded, 46 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         3.79         0.76
      1      0.043       3.79         0.76   
edi fit: project cosio_heidi, 1424 pts fitted / 1424 loaded, 50 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        10.51         4.76
      1      0.075       3.77         0.61   87.1% ↓
      2      0.087       3.77         0.61   
      3      0.105       3.77         0.61   
  done · converged · 5 iters · reduced χ² 0.62 · wR(F²) 4.18 % · 0.24 s


View(html=<254,659 characters>, height=376)

In [48]:
project.display.pattern('heidi_2k5')

View(html=<235,690 characters>, height=480)

Sazonov et al., Acta Cryst. B 65 (2009) 664-675, Table 4, give Co1 (1.20(3), 3.64(2), 0.57(9))
μB and Co2 3.35(2) μB along b. The fitted moments' uncertainties rest on the estimated 2.5 K
uncertainties.

In [49]:
project.display.structure('cosio_2k5')

View(html=<342,057 characters>, height=660)

## Save the project

In [50]:
project.save_as('sc-neut-cwl_cosio-heidi')
project.report.save_html('sc-neut-cwl_cosio-heidi.html')

Saving project to 'sc-neut-cwl_cosio-heidi'
Saving report to 'sc-neut-cwl_cosio-heidi.html'
